# Prodigal Gene Prediction

![Prodigal Gene Prediction](https://proto-bio.github.io/proto-assets/images/tool/prodigal/hero.png)

This notebook demonstrates `run_prodigal_prediction`, which identifies protein-coding genes in prokaryotic DNA sequences using Prodigal. It covers single-sequence prediction in meta mode, detailed ORF annotation inspection, batch processing, and result export in standard bioinformatics formats.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("prodigal")
display_overview("prodigal")
display_docs_section("prodigal", "Background")

## Available tools

In [ ]:
display_available_tools("prodigal")

### `run_prodigal_prediction`

`run_prodigal_prediction` predicts protein-coding genes in prokaryotic DNA sequences using Prodigal via pyrodigal Python bindings. It runs in two modes: meta mode uses pre-trained metagenomic parameters and works on short contigs, incomplete assemblies, or mixed microbial samples; single-genome mode trains on the input sequence and requires at least 100 kb. Each predicted gene is returned as an ORF object with strand, reading frame, amino acid translation, nucleotide sequence, 1-indexed coordinates, GC content, start codon type, ribosome binding site motif and spacer, and partial-gene flags. Batch inputs are processed together and results are indexed by input sequence.

In [ ]:
from proto_tools import ProdigalInput, ProdigalConfig, ProdigalOutput, run_prodigal_prediction

In [ ]:
# Display input docs
display_api_reference("prodigal", "input", "run_prodigal_prediction")

# A real ~1.5 kb E. coli lac operon segment (J01636): the complete lacI gene with its
# ribosome-binding site, the promoter/operator, and the start of lacZ.
sequence = "GACACCATCGAATGGCGCAAAACCTTTCGCGGTATGGCATGATAGCGCCCGGAAGAGAGTCAATTCAGGGTGGTGAATGTGAAACCAGTAACGTTATACGATGTCGCAGAGTATGCCGGTGTCTCTTATCAGACCGTTTCCCGCGTGGTGAACCAGGCCAGCCACGTTTCTGCGAAAACGCGGGAAAAAGTGGAAGCGGCGATGGCGGAGCTGAATTACATTCCCAACCGCGTGGCACAACAACTGGCGGGCAAACAGTCGTTGCTGATTGGCGTTGCCACCTCCAGTCTGGCCCTGCACGCGCCGTCGCAAATTGTCGCGGCGATTAAATCTCGCGCCGATCAACTGGGTGCCAGCGTGGTGGTGTCGATGGTAGAACGAAGCGGCGTCGAAGCCTGTAAAGCGGCGGTGCACAATCTTCTCGCGCAACGCGTCAGTGGGCTGATCATTAACTATCCGCTGGATGACCAGGATGCCATTGCTGTGGAAGCTGCCTGCACTAATGTTCCGGCGTTATTTCTTGATGTCTCTGACCAGACACCCATCAACAGTATTATTTTCTCCCATGAAGACGGTACGCGACTGGGCGTGGAGCATCTGGTCGCATTGGGTCACCAGCAAATCGCGCTGTTAGCGGGCCCATTAAGTTCTGTCTCGGCGCGTCTGCGTCTGGCTGGCTGGCATAAATATCTCACTCGCAATCAAATTCAGCCGATAGCGGAACGGGAAGGCGACTGGAGTGCCATGTCCGGTTTTCAACAAACCATGCAAATGCTGAATGAGGGCATCGTTCCCACTGCGATGCTGGTTGCCAACGATCAGATGGCGCTGGGCGCAATGCGCGCCATTACCGAGTCCGGGCTGCGCGTTGGTGCGGATATCTCGGTAGTGGGATACGACGATACCGAAGACAGCTCATGTTATATCCCGCCGTCAACCACCATCAAACAGGATTTTCGCCTGCTGGGGCAAACCAGCGTGGACCGCTTGCTGCAACTCTCTCAGGGCCAGGCGGTGAAGGGCAATCAGCTGTTGCCCGTCTCACTGGTGAAAAGAAAAACCACCCTGGCGCCCAATACGCAAACCGCCTCTCCCCGCGCGTTGGCCGATTCATTAATGCAGCTGGCACGACAGGTTTCCCGACTGGAAAGCGGGCAGTGAGCGCAACGCAATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGGAATTGTGAGCGGATAACAATTTCACACAGGAAACAGCTATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGGGAAAACCCTGGCGTTACCCAACTTAATCGCCTTGCAGCACATCCCCCTTTCGCCAGCTGGCGTAATAGCGAAGAGGCCCGCACCGATCGCCCTTCCCAACAGTTGCGCAGCCTGAATGGCGAATGGCGCTTTGCCTGGTTTCCGGCACCAGAAGCGGTGCCGGAAA"

inputs = ProdigalInput(input_sequences=sequence)

In [ ]:
# Display config docs
display_api_reference("prodigal", "config", "run_prodigal_prediction")

# Meta mode with bacterial translation table | see docs above for all fields
config = ProdigalConfig(
    meta_mode=True,
    translation_table="bacterial",
    closed_ends=False,
)

In [ ]:
# Run the prediction
result = run_prodigal_prediction(inputs, config)

In [ ]:
# Display output docs
display_api_reference("prodigal", "output", "run_prodigal_prediction")

# Inspect predicted genes
print(f"Found {result.num_orfs} predicted gene(s)")
for orf in result.predicted_orfs[0]:
    print(f"  {orf.id}: {orf.nucleotide_start}-{orf.nucleotide_end} ({orf.strand} strand, frame {orf.frame})")

# Inspect rich annotations on the first predicted ORF
orf = result.predicted_orfs[0][0]
print(f"\nGene annotations:")
print(f"  Start type:     {orf.start_type}")
print(f"  RBS motif:      {orf.rbs_motif}")
print(f"  RBS spacer:     {orf.rbs_spacer}")
print(f"  GC content:     {orf.gc_content:.3f}")
print(f"  Partial (5'/3'): {orf.partial_begin}/{orf.partial_end}")
print(f"  Protein length: {orf.amino_acid_length} aa")
print(f"  Protein:        {orf.amino_acid_sequence[:50]}...")

# Demonstrate batch processing
batch_sequences = [
    "ATGCGTAAATAA" * 50,
    "ATGGCATAA" * 50,
    "ATGAAACGT" * 50,
]
batch_result = run_prodigal_prediction(ProdigalInput(input_sequences=batch_sequences))
print(f"\nBatch: {batch_result.num_orfs} total genes across {len(batch_sequences)} sequences")
print(f"Genes per sequence: {batch_result.num_orfs_per_sequence}")

## Export Results

Predicted genes can be exported to GFF (standard gene annotation format), CSV or JSON for tabular analysis, or FASTA format (FAA for protein sequences, FNA for nucleotide sequences) for downstream tools such as BLAST or multiple sequence alignment.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("prodigal_results", export_path=output_dir, file_format="gff")
print(f"Exported GFF to {output_dir / 'prodigal_results.gff'}")

result.export("prodigal_results", export_path=output_dir, file_format="csv")
print(f"Exported CSV to {output_dir / 'prodigal_results.csv'}")

result.export("prodigal_results", export_path=output_dir, file_format="faa")
print(f"Exported protein FASTA to {output_dir / 'prodigal_results.faa'}")